In [1]:
from models.weight_quantile_vae import *

In [2]:
import torch
from models.weight_quantile_vae import WeightQuantileVAE, ModelConfig, EncoderConfig, ResamplerConfig


def safe_check_3x3(self_attn_mode: str = "full"):
    torch.manual_seed(7)
    device = torch.device("cpu")

    # Small CPU-only sanity check: W is 3x3
    n = 16
    d_in = 3
    d_out = 3

    X = torch.randn(n, d_in, device=device)
    W = torch.randn(d_in, d_out, device=device)

    cfg = ModelConfig(
        k=4,
        k_mlp=8,
        patch_size=2,
        d_tok=16,
        m_lat=4,
        d_lat=16,
        n_heads=4,
        encoder=EncoderConfig(n_row_layers=2, self_attn_mode=self_attn_mode),
        resampler=ResamplerConfig(n_layers=2),
        dropout=0.0,
    )

    model = WeightQuantileVAE(cfg).to(device).eval()
    with torch.no_grad():
        W_hat, kl_loss, aux = model(X, W)

    assert W_hat.shape == (d_in, d_out), f"bad W_hat shape: {tuple(W_hat.shape)}"
    assert kl_loss.ndim == 0, f"kl_loss must be scalar, got ndim={kl_loss.ndim}"
    assert W_hat.device.type == "cpu", f"W_hat expected on cpu, got {W_hat.device}"
    assert torch.isfinite(W_hat).all().item(), "W_hat contains non-finite values"
    assert torch.isfinite(kl_loss).item(), "kl_loss is non-finite"

    print(
        f"[ok] mode={self_attn_mode} | W_hat={tuple(W_hat.shape)} | "
        f"kl={float(kl_loss):.6f} | n_patches={aux['n_patches']}"
    )


for mode in ("full", "cls_only"):
    safe_check_3x3(mode)


[ok] mode=full | W_hat=(3, 3) | kl=1.767084 | n_patches=2
[ok] mode=cls_only | W_hat=(3, 3) | kl=1.316693 | n_patches=2
